# Fine-Tune AI Image Detector on Google Colab
## Using Ateeqq/ai-vs-human-image-detector with Custom Dataset

This notebook fine-tunes a SiGLIP-based image classification model to detect AI-generated vs human images on your custom dataset.

### What You'll Learn:
- Set up the environment and dependencies
- Prepare your custom dataset
- Fine-tune the pretrained model
- Evaluate model performance
- Save and deploy the model

**Estimated Runtime:** 30-60 minutes (depending on dataset size)

---

## 1️⃣ Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q transformers torch torchvision datasets pillow accelerate scikit-learn tensorboard wandb --upgrade
!pip install -q huggingface-hub

print("✅ All dependencies installed!")

In [ ]:
# Check GPU availability
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device: {device}")

if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ Running on CPU - consider enabling GPU in Colab (Runtime > Change runtime type > GPU)")

In [ ]:
# Import libraries
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoImageProcessor,
    SiglipForImageClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset, Image as HFImage

print("✅ All imports successful!")

## 2️⃣ Prepare Your Dataset

### Option A: Upload from Local Folder
If you have images locally, upload them here first.

Expected structure:
```
dataset/
├── ai/
│   ├── image1.jpg
│   ├── image2.png
│   └── ...
└── human/
    ├── image1.jpg
    ├── image2.png
    └── ...
```

In [ ]:
# Mount Google Drive to access your dataset
from google.colab import drive
drive.mount('/content/gdrive')

print("✅ Google Drive mounted!")
print("Navigate to /content/gdrive/My Drive/ to find your files")

In [ ]:
# Configure these variables based on your dataset location
DATASET_PATH = "/content/gdrive/My Drive/dataset"  # Change this to your dataset path
AI_LABEL = "ai"
HUMAN_LABEL = "human"

# Verify dataset structure
def verify_dataset(dataset_path):
    """
    Verify dataset structure and count images
    """
    ai_dir = Path(dataset_path) / AI_LABEL
    human_dir = Path(dataset_path) / HUMAN_LABEL
    
    if not ai_dir.exists() or not human_dir.exists():
        print(f"❌ Dataset structure not found at {dataset_path}")
        print(f"Expected directories: {AI_LABEL}/ and {HUMAN_LABEL}/")
        return False
    
    ai_count = len(list(ai_dir.glob('*.*')))
    human_count = len(list(human_dir.glob('*.*')))
    
    print(f"✅ Dataset verified!")
    print(f"   AI images: {ai_count}")
    print(f"   Human images: {human_count}")
    print(f"   Total: {ai_count + human_count}")
    
    return True

verify_dataset(DATASET_PATH)

In [ ]:
# Load dataset from directory structure
from datasets import load_dataset

# Load using image_classification structure
dataset = load_dataset(
    'imagefolder',
    data_dir=DATASET_PATH,
    split='train'  # If you want to use all data for training
)

print(f"✅ Dataset loaded: {len(dataset)} images")
print(f"Class distribution: {dataset.features['label'].names}")
print(f"Sample: {dataset[0]}")

In [ ]:
# Split dataset into train/validation (80/20 split)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

print(f"✅ Dataset split complete!")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Validation samples: {len(val_dataset)}")

## 3️⃣ Load Pretrained Model

In [ ]:
# Model configuration
MODEL_NAME = "Ateeqq/ai-vs-human-image-detector"

# Load processor and model
print(f"Loading model from: {MODEL_NAME}")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = SiglipForImageClassification.from_pretrained(MODEL_NAME)
model.to(device)

print(f"✅ Model loaded successfully!")
print(f"   Model type: {type(model).__name__}")
print(f"   Number of parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"   Classes: {model.config.id2label}")

## 4️⃣ Prepare Data for Training

In [ ]:
# Define preprocessing function
def preprocess_function(examples):
    """
    Preprocess images for the model
    """
    images = [img.convert('RGB') for img in examples['image']]
    inputs = processor(images=images, return_tensors='pt')
    inputs['labels'] = examples['label']
    return inputs

# Apply preprocessing
print("Preprocessing training dataset...")
processed_train = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=['image']  # Remove original image column after processing
)

print("Preprocessing validation dataset...")
processed_val = val_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=['image']
)

print(f"✅ Preprocessing complete!")
print(f"   Training batch: {processed_train[0]}")

In [ ]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    """
    Compute metrics for evaluation
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("✅ Metrics function defined!")

## 5️⃣ Configure Training Parameters

In [ ]:
# Configure training arguments
training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    overwrite_output_dir=True,
    
    # Training parameters
    num_train_epochs=3,  # Number of training epochs
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    
    # Learning rate and optimization
    learning_rate=5e-5,  # Learning rate (adjust based on performance)
    warmup_steps=100,    # Warmup steps for scheduler
    weight_decay=0.01,   # L2 regularization
    
    # Evaluation and saving
    eval_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",        # Save checkpoint at the end of each epoch
    load_best_model_at_end=True,   # Load best model after training
    metric_for_best_model="f1",    # Use F1 score as best model metric
    greater_is_better=True,
    
    # Logging
    logging_steps=50,
    logging_dir="./logs",
    
    # Hardware
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    gradient_accumulation_steps=1,
    
    # Other
    seed=42,
    dataloader_num_workers=0,  # Set to 0 for Colab compatibility
    remove_unused_columns=False,
)

print("✅ Training arguments configured!")
print(f"   Output directory: {training_args.output_dir}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")

## 6️⃣ Fine-Tune the Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_train,
    eval_dataset=processed_val,
    compute_metrics=compute_metrics,
    tokenizer=processor,  # Passes the image processor safely
)

In [ ]:
# Start training
print("🚀 Starting training...")
train_results = trainer.train()

print(f"\n✅ Training completed!")
print(f"   Final training loss: {train_results.training_loss:.4f}")

## 7️⃣ Evaluate Model Performance

In [ ]:
# Evaluate on validation set
print("Evaluating on validation set...")
eval_results = trainer.evaluate()

print("\n✅ Evaluation Results:")
print(f"   Accuracy:  {eval_results['eval_accuracy']:.4f}")
print(f"   Precision: {eval_results['eval_precision']:.4f}")
print(f"   Recall:    {eval_results['eval_recall']:.4f}")
print(f"   F1 Score:  {eval_results['eval_f1']:.4f}")
print(f"   Loss:      {eval_results['eval_loss']:.4f}")

In [ ]:
# Get predictions for confusion matrix
predictions = trainer.predict(processed_val)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = processed_val['labels']

# Create confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=model.config.id2label.values(),
    yticklabels=model.config.id2label.values()
)
plt.title('Confusion Matrix - Validation Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved!")

## 8️⃣ Save and Push Model

In [ ]:
# Save the fine-tuned model locally
model_save_path = "./my_fine_tuned_detector"
trainer.save_model(model_save_path)
processor.save_pretrained(model_save_path)

print(f"✅ Model saved to {model_save_path}")
print(f"   Files saved:")
for f in os.listdir(model_save_path):
    print(f"     - {f}")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # Login to Hugging Face

trainer.push_to_hub("Ai_image_detector")
print("✅ Model pushed to Hugging Face Hub!")

print("ℹ️  To push to Hugging Face Hub:")
print("   2. Get a token from https://huggingface.co/settings/tokens")
print("   3. Replace 'my-ai-image-detector-v2' with your repo name")

## 9️⃣ Test Inference on New Images

In [ ]:
# Load the fine-tuned model for inference
from transformers import pipeline

# Create inference pipeline with your fine-tuned model
classifier = pipeline(
    "image-classification",
    model=model_save_path,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ Inference pipeline ready!")

In [ ]:
# Test on a sample image
# Replace this with your own image path
test_image_path = "/content/gdrive/My Drive/test_image.jpg"  # Change this

if os.path.exists(test_image_path):
    # Run inference
    results = classifier(test_image_path)
    
    # Display results
    print(f"\n📊 Inference Results for: {os.path.basename(test_image_path)}")
    print("="*50)
    for result in results:
        label = result['label']
        score = result['score']
        print(f"   {label.upper()}: {score*100:.2f}%")
    
    # Display image
    img = Image.open(test_image_path)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Detected: {results[0]['label'].upper()} ({results[0]['score']*100:.1f}%)")
    plt.tight_layout()
    plt.show()
else:
    print(f"❌ Image not found at {test_image_path}")
    print("   Update the path to your test image")

In [ ]:
# Batch inference on multiple images
def batch_infer(image_dir, classifier):
    """
    Run inference on all images in a directory
    """
    results = []
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}
    
    for image_path in Path(image_dir).iterdir():
        if image_path.suffix.lower() in image_extensions:
            try:
                pred = classifier(str(image_path))
                results.append({
                    'filename': image_path.name,
                    'label': pred[0]['label'],
                    'confidence': pred[0]['score']
                })
            except Exception as e:
                print(f"Error processing {image_path.name}: {e}")
    
    return results

# Example: Run batch inference
# batch_results = batch_infer('/content/gdrive/My Drive/test_images/', classifier)
# 
# # Display results
# for result in batch_results:
#     print(f"{result['filename']}: {result['label']} ({result['confidence']*100:.2f}%)")

print("✅ Batch inference function ready!")
print("   Uncomment to run on your test images")

## Summary & Next Steps

### ✅ What You've Done:
1. Set up the environment with all dependencies
2. Loaded your custom dataset
3. Fine-tuned the SiGLIP-based detector on your images
4. Evaluated model performance
5. Saved the fine-tuned model
6. Tested inference on new images

### 🎯 Key Results:
- Model Accuracy: Check the evaluation metrics above
- Confusion Matrix: Shows per-class performance
- Inference Speed: Fast inference ready for production

### 📈 Next Steps:
1. **Improve Performance:**
   - Increase dataset size (more training data = better results)
   - Adjust hyperparameters (learning_rate, num_epochs, batch_size)
   - Augment data (rotation, brightness, contrast, etc.)

2. **Deploy Model:**
   - Push to Hugging Face Hub for easy sharing
   - Use in production with FastAPI or similar
   - Deploy on cloud (AWS, GCP, Azure)

3. **Monitor Performance:**
   - Track accuracy on new data
   - Retrain periodically with fresh data
   - Use A/B testing for model updates

### 📚 Resources:
- [Hugging Face Docs](https://huggingface.co/docs)
- [SiGLIP Paper](https://arxiv.org/abs/2311.17911)
- [Fine-tuning Guide](https://huggingface.co/docs/transformers/training)

---

**Happy training! 🚀**